In [1]:
import time
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [13]:
import zipfile
import pandas as pd

# Zip extract
with zipfile.ZipFile('breast+cancer.zip', 'r') as zip_ref:
    zip_ref.extractall('breast_cancer_data') 

data = pd.read_csv(
    './breast_cancer_data/breast-cancer.data',
    names=[
        'class','age','menopause','tumor-size','inv-nodes',
        'node-caps','deg-malig','breast','breast-quad','irradiat'
    ]
)

print(data.head())


                  class    age menopause tumor-size inv-nodes node-caps  \
0  no-recurrence-events  30-39   premeno      30-34       0-2        no   
1  no-recurrence-events  40-49   premeno      20-24       0-2        no   
2  no-recurrence-events  40-49   premeno      20-24       0-2        no   
3  no-recurrence-events  60-69      ge40      15-19       0-2        no   
4  no-recurrence-events  40-49   premeno        0-4       0-2        no   

   deg-malig breast breast-quad irradiat  
0          3   left    left_low       no  
1          2  right    right_up       no  
2          2   left    left_low       no  
3          2  right     left_up       no  
4          2  right   right_low       no  


In [14]:
data.shape

(286, 10)

In [15]:
data.head()

,class,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,no-recurrence-events,30-39,premeno,30-34,0-2,no,3,left,left_low,no
1,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,right,right_up,no
2,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,left,left_low,no
3,no-recurrence-events,60-69,ge40,15-19,0-2,no,2,right,left_up,no
4,no-recurrence-events,40-49,premeno,0-4,0-2,no,2,right,right_low,no


In [16]:
data['node-caps'].unique()

array(['no', 'yes', '?'], dtype=object)

In [17]:
data[data['node-caps']=='?'].shape[0]


8

In [18]:
data['breast-quad'].unique()


array(['left_low', 'right_up', 'left_up', 'right_low', 'central', '?'],
      dtype=object)

In [19]:
data[data['breast-quad']=='?'].shape[0]


1

In [20]:
data.loc[data['node-caps']=='?','node-caps'] = data['node-caps'].mode().values[0]


In [21]:
data.loc[data['breast-quad']=='?','breast-quad'] = data['breast-quad'].mode().values[0]


In [22]:
y = data['class']
X = data.drop('class',axis=1)

In [23]:
age = ['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']
menopause = ['lt40', 'ge40', 'premeno']
tumor_size = ['0-4', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59']
inv_nodes = ['0-2', '3-5', '6-8', '9-11', '12-14', '15-17', '18-20', '21-23', '24-26', '27-29', '30-32', '33-35', '36-39']

In [24]:
encoder = OrdinalEncoder(categories=[age,menopause,tumor_size,inv_nodes])


In [25]:
X.loc[:,['age','menopause','tumor-size','inv-nodes']] = encoder.fit_transform(X[['age','menopause','tumor-size','inv-nodes']])


In [26]:
X.head(5)


,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,2.0,2.0,6.0,0.0,no,3,left,left_low,no
1,3.0,2.0,4.0,0.0,no,2,right,right_up,no
2,3.0,2.0,4.0,0.0,no,2,left,left_low,no
3,5.0,1.0,3.0,0.0,no,2,right,left_up,no
4,3.0,2.0,0.0,0.0,no,2,right,right_low,no


In [28]:
# Enforce datatypes in our columns:

for cat in X.columns.values:
    if cat in ['node-caps', 'breast', 'breast-quad', 'irradiat']:
        X[cat] = X[cat].astype('category')
    else:
        X[cat] = X[cat].astype('int')

#Finally, convert target into numerical values:

In [29]:
y.unique()

array(['no-recurrence-events', 'recurrence-events'], dtype=object)

In [30]:
y.mask(y=='no-recurrence-events', 0, inplace=True)
y.mask(y=='recurrence-events', 1, inplace=True)

In [31]:
y.unique()

array([0, 1], dtype=object)

In [32]:
y.shape[0]

286

In [33]:
y[y==0].shape[0]

201

In [34]:
y[y==1].shape[0]

85

In [35]:
def balanced_train_test_split(X, y, test_size):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    # balance the classes of the training data
    D_train = X_train.copy()
    D_train['y'] = y_train 
    D0 = D_train[D_train.y == 0]
    D1 = D_train[D_train.y == 1]
    n_samples = int(X_train.shape[0]/2)
    D0 = D0.sample(n=n_samples,replace=True,random_state=42)
    D1 = D1.sample(n=n_samples,replace=True,random_state=42)
    D_train = pd.concat([D0,D1])
    y_train = D_train['y']
    X_train = D_train.drop('y',axis=1)
    return (X_train,X_test,y_train,y_test)

# XGBoost

In [36]:
X_train, X_test, y_train, y_test = balanced_train_test_split(X, y, test_size=0.2)


In [37]:
start_time = time.time()
model = XGBClassifier(
    learning_rate=0.01, 
    max_depth=5, 
    n_estimators=500, 
    enable_categorical=True, 
    random_state=42
)
model.fit(X_train,y_train)
print(f"training time duration: {time.time() - start_time:.2f}")

training time duration: 0.52


In [38]:
y_test = y_test.values.tolist()
y_pred = model.predict(X_test).tolist()
print(f'accuracy score: {accuracy_score(y_test,y_pred):.2f}')
print(f'precision score: {precision_score(y_test,y_pred):.2f}')
print(f'recall score: {recall_score(y_test,y_pred):.2f}')
print(f'f1 score: {f1_score(y_test,y_pred):.2f}')

accuracy score: 0.66
precision score: 0.52
recall score: 0.62
f1 score: 0.57


# Catboost

In [39]:
start_time = time.time()
model = CatBoostClassifier(
    learning_rate=0.01, 
    max_depth=5, 
    n_estimators=500, 
    cat_features=['node-caps', 'breast', 'breast-quad', 'irradiat'], 
    verbose=0,
    random_state=42
)
model.fit(X_train,y_train)
print(f"training time duration: {time.time() - start_time:.2f}")

training time duration: 9.78


In [40]:
y_pred = model.predict(X_test).tolist()
print(f'accuracy score: {accuracy_score(y_test,y_pred):.2f}')
print(f'precision score: {precision_score(y_test,y_pred):.2f}')
print(f'recall score: {recall_score(y_test,y_pred):.2f}')
print(f'f1 score: {f1_score(y_test,y_pred):.2f}')

accuracy score: 0.67
precision score: 0.55
recall score: 0.57
f1 score: 0.56


# LightGBM

In [41]:
# OHE the categorical features, then do train-test split
Xohe = pd.get_dummies(X,columns=['node-caps', 'breast', 'breast-quad', 'irradiat']).astype(int)
X_train, X_test, y_train, y_test = balanced_train_test_split(Xohe, y, test_size=0.2)

In [42]:
start_time = time.time()
model = LGBMClassifier(learning_rate=0.01, max_depth=5, n_estimators=500, verbose=-1, random_state=42)
model.fit(X_train,y_train.astype(int))
print(f"training time duration: {time.time() - start_time:.2f}")

training time duration: 2.80


In [43]:
y_test = y_test.values.tolist()
y_pred = model.predict(X_test).tolist()
print(f'accuracy score: {accuracy_score(y_test,y_pred):.2f}')
print(f'precision score: {precision_score(y_test,y_pred):.2f}')
print(f'recall score: {recall_score(y_test,y_pred):.2f}')
print(f'f1 score: {f1_score(y_test,y_pred):.2f}')

accuracy score: 0.67
precision score: 0.54
recall score: 0.62
f1 score: 0.58


# Adaboost

In [44]:
start_time = time.time()
model = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=5),
    algorithm='SAMME',
    learning_rate=0.01, 
    n_estimators=500, 
    random_state=42
)
model.fit(X_train,y_train.astype(int))
print(f"training time duration: {time.time() - start_time:.2f}")

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


training time duration: 0.68


In [47]:
y_pred = model.predict(X_test).tolist()
print(f'accuracy score: {accuracy_score(y_test,y_pred):.2f}')
print(f'precision score: {precision_score(y_test,y_pred):.2f}')
print(f'recall score: {recall_score(y_test,y_pred):.2f}')
print(f'f1 score: {f1_score(y_test,y_pred):.2f}')

accuracy score: 0.71
precision score: 0.59
recall score: 0.62
f1 score: 0.60


# GBM

In [48]:
start_time = time.time()
model = GradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.01, 
    n_estimators=500, 
    random_state=42
)
model.fit(X_train,y_train.astype(int))
print(f"training time duration: {time.time() - start_time:.2f}")

training time duration: 0.81


In [49]:
y_pred = model.predict(X_test).tolist()
print(f'accuracy score: {accuracy_score(y_test,y_pred):.2f}')
print(f'precision score: {precision_score(y_test,y_pred):.2f}')
print(f'recall score: {recall_score(y_test,y_pred):.2f}')
print(f'f1 score: {f1_score(y_test,y_pred):.2f}')

accuracy score: 0.62
precision score: 0.48
recall score: 0.57
f1 score: 0.52
